# Fruit Freshness Classifier


**Supported fruits:** Banana · Apples · Oranges

**Models:** Custom CNN from scratch + MobileNetV3-Large transfer learning

---
### ⚙️ Before running: set runtime to GPU
`Runtime → Change runtime type → T4 GPU`

# 1 Check GPU & Install Dependencies

In [1]:
import torch

if torch.cuda.is_available():
    print("GPU Available")
else:
    print('No GPU')

!pip install -q kaggle scikit-learn seaborn onnx tqdm onnxscript
print('\n installs ready')


KeyboardInterrupt: 

##2 Mount Google Drive
Saves datasets and checkpoints so they survive session restarts.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/fruit_classifier'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')


##3 Kaggle Credentials
Upload your `kaggle.json` (from kaggle.com → Account → Create New API Token).

In [ ]:
from google.colab import files
import os, shutil

print('Upload your kaggle.json file:')
uploaded = files.upload()

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials configured')


## 4 Utilities (data_utils)

> Add blockquote



In [ ]:
from pathlib import Path # Helps manage file paths
from torch.utils.data import DataLoader  # PyTorch utility for batching and loading data
from torchvision import datasets, transforms # Built-in datasets and image transformations



# Define the base fruit categories
FRUITS = ['banana', 'apple', 'citrus']


# Create a full list of class labels:
ALL_CLASSES = (
    ['not_fruit']
    + [f'{f}_fresh'  for f in FRUITS]
    + [f'{f}_rotten' for f in FRUITS]
)

#image size (specifically for transfer learning)
IMG_SIZE = 224


# Transformations applied to training images (data augmentation)
TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


# Transformations for validation/test (NO augmentation)
EVAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])



#  get_loaders reates PyTorch DataLoaders for training, validation, and test datasets.

# Args:
#         data_root (str): Root directory containing 'train', 'val', 'test' folders
#         batch_size (int): Number of samples per batch
#         num_workers (int): Number of subprocesses for data loading

# Returns:
#         train_loader, val_loader, test_loader, class_to_idx

def get_loaders(data_root='data', batch_size=32, num_workers=2):

    root = Path(data_root)

    # Load datasets using folder structure:
    train_ds = datasets.ImageFolder(str(root / 'train'), transform=TRAIN_TRANSFORMS)
    val_ds   = datasets.ImageFolder(str(root / 'val'),   transform=EVAL_TRANSFORMS)
    test_ds  = datasets.ImageFolder(str(root / 'test'),  transform=EVAL_TRANSFORMS)

    #Create DataLoaders (handles batching, shuffling, parallel loading)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)


    # Print dataset summary for sanity check
    print(f'Classes ({len(train_ds.classes)}): {train_ds.classes}')
    print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
    return train_loader, val_loader, test_loader, train_ds.class_to_idx

print('utils defined')


## 5 Download & Prepare Dataset
Downloads ~3 GB from Kaggle and splits into train/val/test.  
Skips automatically if already done (safe to re-run).

In [ ]:
import os, random, shutil, subprocess
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image



SEED        = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

# Directory
DATA_ROOT   = Path('data')  # Final structured dataset
RAW_DIR     = Path('raw_downloads') # Where raw datasets are downloaded


# Define classes
CLASSES = ['not_fruit'] + [f'{f}_fresh' for f in FRUITS] + [f'{f}_rotten' for f in FRUITS]


# Kaggle datasets to download
KAGGLE_DATASETS = [
    'sriramr/fruits-fresh-and-rotten-for-classification',
    'ifigotin/imagenetmini-1000',
]

# Mapping raw dataset folder names to labels
RAW_LABEL_MAP = {
    'freshbanana':   'banana_fresh',
    'rottenbanana':  'banana_rotten',
    'freshapples':   'apple_fresh',
    'rottenapples':  'apple_rotten',
    'freshoranges':  'citrus_fresh',
    'rottenoranges': 'citrus_rotten',
}

# Downloads datasets from Kaggle
def download_datasets():
    RAW_DIR.mkdir(exist_ok=True)
    for ds in KAGGLE_DATASETS:
        dest = RAW_DIR / ds.split('/')[1]
        if dest.exists():
            print(f'[skip] {ds}')
            continue
        print(f'[download] {ds} ...')
        subprocess.run(['kaggle', 'datasets', 'download', '-d', ds,
                        '--unzip', '-p', str(dest)], check=True)
    print('All datasets downloaded')

 # Recursively collect images from a directory and assign them a label.
def _collect_images_from_dir(src, label, collected):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    for p in src.rglob('*'):
        if p.suffix.lower() in exts and p.is_file():
            collected.setdefault(label, []).append(p)

#collect images from all raw datasets
def collect_all_images():
    collected = {}
    ds1 = RAW_DIR / 'fruits-fresh-and-rotten-for-classification'
    if ds1.exists():
        for folder in ds1.rglob('*'):
            if folder.is_dir():
                key = folder.name.lower().replace(' ', '')
                if key in RAW_LABEL_MAP:
                    _collect_images_from_dir(folder, RAW_LABEL_MAP[key], collected)
    ds2 = RAW_DIR / 'imagenetmini-1000'
    if ds2.exists():
        imagenet_imgs = []
        for p in ds2.rglob('*'):
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and p.is_file():
                imagenet_imgs.append(p)
                if len(imagenet_imgs) >= 2000:
                    break
        collected.setdefault('not_fruit', []).extend(imagenet_imgs)
    print('Collected images per class:')
    for cls in CLASSES:
        print(f'  {cls:30s}: {len(collected.get(cls, []))}')
    return collected


# Splits images into train/val/test sets and copies them into
# a structured directory format compatible with ImageFolder.

def split_and_copy(collected):
    random.seed(SEED)
    for split in ['train', 'val', 'test']:
        for cls in CLASSES:
            (DATA_ROOT / split / cls).mkdir(parents=True, exist_ok=True)
    for cls in CLASSES:
        imgs = collected.get(cls, [])
        random.shuffle(imgs)
        n       = len(imgs)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)
        splits  = {
            'train': imgs[:n_train],
            'val':   imgs[n_train: n_train + n_val],
            'test':  imgs[n_train + n_val:],
        }
        for split_name, paths in splits.items():
            dest_dir = DATA_ROOT / split_name / cls
            for src in tqdm(paths, desc=f'{split_name}/{cls}', leave=False):
                dest = dest_dir / src.name
                if dest.exists():
                    dest = dest_dir / (src.stem + f'_{random.randint(0,99999)}' + src.suffix)
                try:
                    Image.open(src).convert('RGB').save(dest)
                except Exception:
                    pass
    print('\n Dataset split complete')
    for split in ['train', 'val', 'test']:
        total = sum(len(list((DATA_ROOT / split / cls).iterdir()))
                    for cls in CLASSES if (DATA_ROOT / split / cls).exists())
        print(f'  {split:6s}: {total} images')

#Run the functions
download_datasets()
collected = collect_all_images()
split_and_copy(collected)


## 6 Define Both Models

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

# Model A: Custom CNN from scratch

#  A basic convolutional block:
#         - Conv2D → BatchNorm → ReLU
#         - Optional MaxPooling for downsampling

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]

        # Optionally add spatial downsampling
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

#Custom CNN for fruit classification.
class FruitCNNScratch(nn.Module):
    def __init__(self, num_classes=11):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32,  pool=True),
            ConvBlock(32,  64,  pool=True),
            ConvBlock(64,  128, pool=True),
            ConvBlock(128, 256, pool=False),
            ConvBlock(256, 256, pool=True),
            ConvBlock(256, 512, pool=False),
            ConvBlock(512, 512, pool=True),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
        self._init_weights()


    #initialize weights
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    #forward pass
    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


# Model B: MobileNetV3 Transfer Learning


#Loads a pretrained MobileNetV3 model and adapts it for classification.

def build_transfer_model(num_classes, freeze_backbone=True):
    model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    if freeze_backbone:
        for name, param in model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad_(False)
    return model

#Unfreezes all layers in the model
def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad_(True)

#Sanity Checks

print('Models defined')
s = FruitCNNScratch(11)
t = build_transfer_model(11, freeze_backbone=False)
print(f'   Scratch CNN params  : {sum(p.numel() for p in s.parameters()):>10,}')
print(f'   MobileNetV3 params  : {sum(p.numel() for p in t.parameters()):>10,}')


## 7 Training Loop Utilities

In [ ]:
import time
import torch
import torch.nn as nn

def train_one_epoch(model, loader, criterion, optimizer, device, scaler):

    # Runs one full training epoch:
    # - Forward pass
    # - Loss computation
    # - Backpropagation (with mixed precision)
    # - Gradient clipping
    # - Parameter update


    model.train()  # Set model to training mode (enables dropout, batchnorm updates)

    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        # Move data to device (CPU or GPU)
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()  # Reset gradients

        # Mixed precision context (only active on CUDA)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(imgs)           # Forward pass
            loss    = criterion(outputs, labels)  # Compute loss

        # Backpropagation with gradient scaling (prevents underflow in FP16)
        scaler.scale(loss).backward()

        # Unscale gradients before clipping
        scaler.unscale_(optimizer)

        # Clip gradients to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Optimizer step (scaled)
        scaler.step(optimizer)
        scaler.update()  # Update scaling factor

        # Accumulate loss (multiply by batch size for proper averaging)
        running_loss += loss.item() * imgs.size(0)

        # Compute accuracy
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += imgs.size(0)

    # Return average loss and accuracy
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Evaluates the model on validation/test data:
    - No gradient computation (faster, less memory)
    - No parameter updates
    """
    model.eval()  # Set model to evaluation mode (disables dropout, fixes batchnorm)

    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        # Mixed precision (still useful for speed on GPU)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += imgs.size(0)

    # Return average loss and accuracy
    return running_loss / total, correct / total


def run_training_loop(model, train_loader, val_loader, criterion, optimizer,
                      scheduler, scaler, epochs, device, save_path,
                      best_val_acc, label):

    # Full training loop:
    # - Trains for multiple epochs
    # - Evaluates on validation set each epoch
    # - Tracks best model based on validation accuracy
    # - Saves best model checkpoint

    # Count trainable parameters
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Print header
    print(f'\n{chr(9472)*55}')
    print(f'  {label}')
    print(f'  Trainable params: {trainable:,}')
    print(f'{chr(9472)*55}')
    print(f'{"Epoch":>6}  {"Tr Loss":>8}  {"Tr Acc":>7}  {"Vl Loss":>8}  {"Vl Acc":>7}  {"Time":>6}')
    print('-' * 55)

    # Epoch loop
    for epoch in range(1, epochs + 1):
        t0 = time.time()  # Start timing

        # Train for one epoch
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scaler
        )

        # Evaluate on validation set
        vl_loss, vl_acc = evaluate(
            model, val_loader, criterion, device
        )

        # Step learning rate scheduler
        scheduler.step()

        elapsed = time.time() - t0  # Time taken for epoch

        # Print metrics
        print(f'{epoch:>6}  {tr_loss:>8.4f}  {tr_acc:>6.2%}  {vl_loss:>8.4f}  {vl_acc:>6.2%}  {elapsed:>5.1f}s')

        # Save best model based on validation accuracy
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc

            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'val_acc': vl_acc
            }, save_path)

            print(f'Best saved ({vl_acc:.2%})')

    return best_val_acc


print('Training utilities defined')

# 8 Load Data

In [ ]:
# Select device: use GPU if available, otherwise fall back to CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Number of samples per batch during training/evaluation
BATCH_SIZE = 32

# Load data using previously defined utility function
# Returns:
# - DataLoaders for train, validation, and test sets
# - Mapping from class names → integer indices

train_loader, val_loader, test_loader, class_to_idx = get_loaders(
    data_root='data',
    batch_size=BATCH_SIZE,
    num_workers=2  # Number of parallel workers for data loading
)

# Total number of classes (derived from dataset)
NUM_CLASSES = len(class_to_idx)

# Reverse mapping: index → class name (useful for predictions/visualization)
idx_to_class = {v: k for k, v in class_to_idx.items()}

# Print basic setup info for verification
print(f'\nDevice     : {DEVICE}')
print(f'Num classes: {NUM_CLASSES}')


## 9 Train Model A: Custom CNN from Scratch
~40 epochs max


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path

# Training configuration (Scratch CNN)

SCRATCH_EPOCHS = 40      # Number of training epochs
SCRATCH_LR     = 1e-3    # Initial learning rate

# Ensure directory for saving models exists
Path('models').mkdir(exist_ok=True)

# Path to save the best model checkpoint
scratch_save = Path('models/scratch_best.pth')

# Loss function:
# CrossEntropy with label smoothing (helps regularization and prevents overconfidence)
criterion_sc = nn.CrossEntropyLoss(label_smoothing=0.1)

# Initialize custom CNN and move to device (CPU/GPU)
scratch_model = FruitCNNScratch(num_classes=NUM_CLASSES).to(DEVICE)

# Optimizer: AdamW (Adam with decoupled weight decay for better generalization)
optimizer_sc = optim.AdamW(
    scratch_model.parameters(),
    lr=SCRATCH_LR,
    weight_decay=1e-4
)

# Learning rate scheduler:
# Cosine annealing gradually reduces LR following a cosine curve
scheduler_sc = CosineAnnealingLR(
    optimizer_sc,
    T_max=SCRATCH_EPOCHS,  # Full cycle = total epochs
    eta_min=1e-6           # Minimum learning rate
)

# Gradient scaler for mixed precision training (only active on CUDA)
scaler_sc = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))


# Train the model

best_scratch = run_training_loop(
    scratch_model,
    train_loader,
    val_loader,
    criterion_sc,
    optimizer_sc,
    scheduler_sc,
    scaler_sc,
    SCRATCH_EPOCHS,
    DEVICE,
    scratch_save,
    0.0,  # Initial best validation accuracy
    'Custom CNN - Training from Scratch'
)

# ------ Load best model and evaluate -----


# Load saved checkpoint (best validation performance)
ckpt = torch.load(scratch_save, map_location=DEVICE)

# Restore model weights
scratch_model.load_state_dict(ckpt['model_state'])

# Evaluate on test set
te_loss, te_acc = evaluate(
    scratch_model,
    test_loader,
    criterion_sc,
    DEVICE
)

# Print final test performance
print(f'\nScratch CNN  -- Test Acc: {te_acc:.2%}  |  Test Loss: {te_loss:.4f}')


## 10 Train Model B: MobileNetV3 (Transfer Learning)
Two-phase: warm-up head (5 epochs) then full fine-tune (30 epochs max).

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path


# Training configuration (Transfer Learning)


HEAD_EPOCHS = 5     # Phase 1: train only classifier (head)
FT_EPOCHS   = 30    # Phase 2: fine-tune entire model

LR_HEAD = 1e-3      # Learning rate for head training
LR_FT   = 1e-4      # Base learning rate for fine-tuning

# Ensure model directory exists
Path('models').mkdir(exist_ok=True)

# Path to save best transfer learning model
transfer_save = Path('models/transfer_best.pth')

# Loss function with label smoothing
criterion_tr = nn.CrossEntropyLoss(label_smoothing=0.1)

# Gradient scaler for mixed precision (GPU only)
scaler_tr = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))


# Phase 1: Train classifier head only (backbone frozen)


# Build model with frozen backbone
transfer_model = build_transfer_model(NUM_CLASSES, freeze_backbone=True).to(DEVICE)

# Optimizer only updates parameters that require gradients (i.e., classifier)
opt_head = optim.AdamW(
    filter(lambda p: p.requires_grad, transfer_model.parameters()),
    lr=LR_HEAD,
    weight_decay=1e-4
)

# Cosine annealing scheduler for head training
sched_head = CosineAnnealingLR(
    opt_head,
    T_max=HEAD_EPOCHS,
    eta_min=1e-6
)

# Train phase 1
best_transfer = run_training_loop(
    transfer_model,
    train_loader,
    val_loader,
    criterion_tr,
    opt_head,
    sched_head,
    scaler_tr,
    HEAD_EPOCHS,
    DEVICE,
    transfer_save,
    0.0,  # Initial best validation accuracy
    'Phase 1 - Head warm-up'
)


# Phase 2: Full fine-tuning (unfreeze backbone)

# Unfreeze all layers for full training
unfreeze_all(transfer_model)

# Separate parameters into backbone and classifier (head)
backbone_params = [
    p for n, p in transfer_model.named_parameters()
    if 'classifier' not in n
]

head_params = [
    p for n, p in transfer_model.named_parameters()
    if 'classifier' in n
]

# Optimizer with differential learning rates:
# - Lower LR for pretrained backbone
# - Higher LR for classifier
opt_ft = optim.AdamW([
    {'params': backbone_params, 'lr': LR_FT * 0.1},
    {'params': head_params,     'lr': LR_FT},
], weight_decay=1e-4)

# Scheduler for fine-tuning phase
sched_ft = CosineAnnealingLR(
    opt_ft,
    T_max=FT_EPOCHS,
    eta_min=1e-7
)

# Train phase 2 (continue from best validation accuracy)
best_transfer = run_training_loop(
    transfer_model,
    train_loader,
    val_loader,
    criterion_tr,
    opt_ft,
    sched_ft,
    scaler_tr,
    FT_EPOCHS,
    DEVICE,
    transfer_save,
    best_transfer,  # carry over best accuracy
    'Phase 2 - Full fine-tuning (backbone unfrozen)'
)


# Load best checkpoint
ckpt = torch.load(transfer_save)

# Store class mapping for later inference use
ckpt['class_to_idx'] = class_to_idx

# Save updated checkpoint
torch.save(ckpt, transfer_save)


# ------ Load best model and evaluate -------


# Restore best weights
transfer_model.load_state_dict(ckpt['model_state'])

# Evaluate on test set
te_loss, te_acc = evaluate(
    transfer_model,
    test_loader,
    criterion_tr,
    DEVICE
)

# Print final performance
print(f'\n Transfer model -- Test Acc: {te_acc:.2%}  |  Test Loss: {te_loss:.4f}')


## 11 Export Models (TorchScript + ONNX)

In [ ]:
dummy = torch.zeros(1, 3, 224, 224, device=DEVICE)

# Export Scratch CNN to onnx
scratch_model.eval()
scripted = torch.jit.script(scratch_model)
scripted.save('models/scratch_scripted.pt')
torch.onnx.export(scratch_model, dummy, 'models/scratch_model.onnx',
                  input_names=['image'], output_names=['logits'],
                  dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
                  opset_version=17)
print('Scratch CNN: models/scratch_scripted.pt + models/scratch_model.onnx')

# Export Transfer model to onnx
transfer_model.eval()
traced = torch.jit.trace(transfer_model, dummy)
traced.save('models/transfer_scripted.pt')
torch.onnx.export(transfer_model, dummy, 'models/transfer_model.onnx',
                  input_names=['image'], output_names=['logits'],
                  dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
                  opset_version=17)
print('Transfer:    models/transfer_scripted.pt + models/transfer_model.onnx')
print('\n exports complete')


# 12  Evaluation: Classification Report + Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path


# Select which model to evaluate

# Change to transfer_model / 'transfer' to evaluate the other model
EVAL_MODEL      = scratch_model
EVAL_MODEL_NAME = 'scratch'

@torch.no_grad()
def get_predictions(model, loader, device):
    """
    Runs inference on a dataset and collects:
    - True labels
    - Predicted labels
    """
    all_preds, all_labels = [], []

    model.eval()  # Set model to evaluation mode

    for imgs, labels in loader:
        imgs = imgs.to(device)

        # Get predicted class indices (argmax over logits)
        preds = model(imgs).argmax(1).cpu().numpy()

        # Store predictions and ground truth
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds)



# Generate predictions

# Map indices → class names (for readable reports)
class_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

# Get predictions on test set
labels, preds = get_predictions(EVAL_MODEL, test_loader, DEVICE)


# Multi-class classification report
print('-- Per-class Classification Report --')
print(classification_report(
    labels,
    preds,
    target_names=class_names,
    digits=4
))



# Binary classification: Fruit vs Not-Fruit

# Get index for 'not_fruit'
not_fruit_idx = class_to_idx['not_fruit']

# Convert to binary labels:
# 0 = not_fruit, 1 = fruit
bin_labels = (labels != not_fruit_idx).astype(int)
bin_preds  = (preds  != not_fruit_idx).astype(int)

print('-- Binary: Fruit vs Not-Fruit --')
print(classification_report(
    bin_labels,
    bin_preds,
    target_names=['not_fruit', 'is_fruit'],
    digits=4
))

# Binary classification: Fresh vs Rotten (only fruit images)

# Get indices corresponding to rotten classes
rotten_idxs = {
    v for k, v in class_to_idx.items()
    if 'rotten' in k
}

# Filter only fruit samples (exclude not_fruit)
fruit_mask = labels != not_fruit_idx

# Convert labels:
# 0 = fresh, 1 = rotten
rot_labels = np.array([
    1 if l in rotten_idxs else 0
    for l in labels[fruit_mask]
])

rot_preds = np.array([
    1 if p in rotten_idxs else 0
    for p in preds[fruit_mask]
])

print('-- Binary: Fresh vs Rotten (fruit images only) --')
print(classification_report(
    rot_labels,
    rot_preds,
    target_names=['fresh', 'rotten'],
    digits=4
))

# Confusion Matrix Visualization

# Ensure output directory exists
Path('outputs').mkdir(exist_ok=True)

# Compute confusion matrix
cm = confusion_matrix(labels, preds)

# Create heatmap
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax
)

# Labels and title
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix - {EVAL_MODEL_NAME}')

# Improve readability of x-axis labels
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Save figure
cm_path = f'outputs/{EVAL_MODEL_NAME}_confusion_matrix.png'
plt.savefig(cm_path, dpi=150)

# Display plot
plt.show()

print(f'Saved to {cm_path}')


## 13 Predict on a New Image
Upload any image and run it through the trained model.

In [ ]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder("/content/drive/MyDrive/fruit_classifier/data/train")
idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import torch  # still used for softmax

print('Upload an image to classify:')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# Load ONNX model
ONNX_MODEL_PATH = "/content/drive/MyDrive/fruit_classifier/models/transfer_model.onnx"  # change if needed
session = ort.InferenceSession(ONNX_MODEL_PATH, providers=["CPUExecutionProvider"])

input_name = session.get_inputs()[0].name

@torch.no_grad()
def predict_single_onnx(session, path, idx_to_class):
    img = Image.open(path).convert('RGB')

    # Apply same transforms as training
    tensor = EVAL_TRANSFORMS(img).unsqueeze(0)  # shape: (1, C, H, W)

    # Convert to numpy for ONNX
    input_array = tensor.cpu().numpy().astype(np.float32)

    # Run inference
    outputs = session.run(None, {input_name: input_array})
    logits = torch.tensor(outputs[0])  # convert back to tensor

    probs = torch.softmax(logits, dim=1)[0]
    top5 = probs.topk(5)

    raw_class = idx_to_class[probs.argmax().item()]
    confidence = probs.max().item()

    if raw_class == 'not_fruit':
        is_fruit_r = False
        fruit_type = None
        condition = None
    else:
        is_fruit_r = True
        parts = raw_class.rsplit('_', 1)
        fruit_type = parts[0].replace('_', ' ').title()
        condition = parts[1].capitalize() if len(parts) > 1 else 'Unknown'

    return dict(
        raw_class=raw_class,
        confidence=confidence,
        is_fruit=is_fruit_r,
        fruit_type=fruit_type,
        condition=condition,
        top5=[(idx_to_class[i.item()], p.item())
              for i, p in zip(top5.indices, top5.values)]
    ), img


# Run inference
result, img = predict_single_onnx(session, img_path, idx_to_class)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img)
axes[0].axis('off')
axes[0].set_title('Input Image')

classes = [c for c, _ in result['top5']]
probs_v = [p for _, p in result['top5']]

colors = [
    '#2ecc71' if 'fresh' in c else
    '#e74c3c' if 'rotten' in c else
    '#95a5a6'
    for c in classes
]

axes[1].barh(classes[::-1], probs_v[::-1], color=colors[::-1])
axes[1].set_xlim(0, 1)
axes[1].set_xlabel('Probability')
axes[1].set_title('Top-5 Predictions')

plt.tight_layout()
plt.show()

print(f"\n{'='*40}")
print(f"  Is fruit?  : {'Yes' if result['is_fruit'] else 'No'}")
if result['is_fruit']:
    print(f"  Fruit type : {result['fruit_type']}")
    print(f"  Condition  : {result['condition']}")
print(f"  Confidence : {result['confidence']:.1%}")
print(f"{'='*40}")

## 14 Download Trained Models

In [ ]:
from google.colab import files
import os

model_files = [
    'models/transfer_best.pth',
    'models/transfer_scripted.pt',
    'models/transfer_model.onnx',
    'models/scratch_best.pth',
    'models/scratch_scripted.pt',
    'models/scratch_model.onnx',
]

for f in model_files:
    if os.path.exists(f):
        print(f'Downloading {f} ...')
        files.download(f)
    else:
        print(f'[skip] {f} not found')
